<a href="https://colab.research.google.com/github/outsider8946/maxidom-hack/blob/dolor/maxidom_hack_filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Init

In [1]:
import json

raw = json.load(open("/Users/dolor.src/Downloads/products_final.json"))
data = raw["products"]

In [2]:
import numpy as np

cats = [list(item["categories"].values())[0] for item in data]
un_cats = np.unique(cats)

len(un_cats), len(cats)

(2375, 88790)

In [3]:
from frida_decisions import AsyncJudge

judge = AsyncJudge.from_pretrained("ai-forever/FRIDA-Decisions", device="mps")

request = {
    "state": "Здравствуйте, у меня не приходит код подтверждения уже час.",
    "questions": {
        "topic": {
            "type": "choice",
            "instructions": "К какой теме относится обращение?",
            "criteria": {
                "login": "вход в аккаунт, коды подтверждения, пароли",
                "payment": "оплата, списания, возвраты",
                "delivery": "доставка заказа",
            },
        },
        "urgency": {
            "type": "score",
            "instructions": "Оцени срочность обращения.",
            "criteria": ["низкая", "средняя", "высокая"],
        },
        "spam": {
            "type": "noul",
            "instructions": "Является ли сообщение спамом?",
            "criteria": {
                "true": "реклама, чужие ссылки, просьба перевести деньги",
                "false": "вопрос или жалоба по нашему сервису",
            },
        },
    },
}

W1025 19:18:40.265000 4277 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

In [5]:
r = await judge.judge_batch([request])
r

[{'model': 'FRIDA-Decisions',
  'answers': {'topic': {'type': 'choice',
    'probabilities': {'login': 0.9988270183781592,
     'payment': 0.00062398738883953,
     'delivery': 0.0005489942330011962},
    'confidence': 0.990990028472675,
    'choice': 'login'},
   'urgency': {'type': 'score',
    'probabilities': {'0': 0.5701981214191202,
     '1': 0.08609470158403032,
     '2': 0.34370717699684944},
    'confidence': 0.18213277991456833,
    'score': 0.7735090555777292,
    'legend': {'0': 'низкая', '1': 'средняя', '2': 'высокая'}},
   'spam': {'type': 'noul', 'noul': 0.002947062206267041}},
  'margins': {'topic': {'login': 9.339272499084473,
    'payment': 1.9610657691955566,
    'delivery': 1.8330235481262207},
   'urgency': {'0': -0.16687995195388794,
    '1': -2.0574159622192383,
    '2': -0.6730737686157227},
   'spam': {'true': 0.8200207352638245, 'false': 6.644015789031982}},
  'usage': {'state_tokens': 12,
   'state_truncated': False,
   'questions': 3,
   'candidates': 8,
   

# DB annotate (unrelevant filter)

In [6]:
def get_noul_request(cat):
    return {
        "state": f"Категория каталога: {cat}",
        "questions": {
            "constructions": {
                "type": "noul",
                "instructions": "Относится ли эта категория товаров к строительству, ремонту или отделке помещений?",
                "criteria": {
                    "true": (
                        "стройматериалы и отделка: плитка, керамогранит, ламинат, линолеум, "
                        "краска, эмаль, грунтовка, обои, ГКЛ, штукатурка, шпаклёвка, "
                        "сухие смеси, цемент, кирпич, утеплитель, гидроизоляция, "
                        "электрика (кабель, розетки, автоматы), сантехника (трубы, смесители), "
                        "плинтусы, наличники, двери, окна, крепёж, строительный инструмент"
                    ),
                    "false": (
                        "не относится к стройке: растения и кашпо, косметика и гигиена, "
                        "одежда и обувь, зоотовары, продукты питания, мебель, "
                        "декор и текстиль, игрушки, канцтовары, автотовары, сад и дача, "
                        "бытовая техника, электроника"
                    ),
                },
            }
        },
    }


def get_choice_request(cat, samples):
    samples_descr = []
    for sample in samples:
        descr = sample.get("marketing_description") or sample.get("name")
        samples_descr.append(descr)
    descr = '\n'.join(samples_descr)
    state = (
        f"Категория каталога: {cat}\n"
        f"Примеры товаров из каталога:\n{descr}"
    )
    return {
        "state": state,
        "questions": {
            "construction": {
                "type": "choice",
                "instructions": "Как эта категория относится к ремонту помещений?",
                "criteria": {
                    "core": "материалы и работы: плитка, краска, обои, ГКЛ, кабель, розетки, автоматы, трубы, смесители, сухие смеси, плинтусы, светильники для монтажа",
                    "adjacent": "сопутствующее ремонту: инструмент, крепёж, расходники, встраиваемая техника, кондиционеры, бойлеры, вентиляция, спецодежда, защита",
                    "excluded": "не относится к ремонту: отдельно стоящая бытовая техника, электроника, растения, косметика, одежда, зоо, продукты, мебель, декор, игрушки",
                },
            }
        },
    }

In [7]:
target_cat = "Трафареты на окна"
products = []
for item in data:
    catalog_cat = list(item["categories"].values())[0]
    if catalog_cat == target_cat:
        products.append(item)

len(products)

2

In [7]:
products

[{'id': 107733,
  'cod_good': '1000739842',
  'name': 'трафарет Новогодний 44,5х59,5см белый в асс-те',
  'marketing_description': None,
  'url_key': '/catalog/trafarety-na-okna/1000739842/',
  'price': 0,
  'currency': 'RUB',
  'brand': {'id': 37, 'name': 'KAEMINGK'},
  'collection': None,
  'country': {'id': 1881, 'name': 'Китай'},
  'categories': {'225': 'Трафареты на окна'},
  'unit': None,
  'description': 'Название: Новогодний<br>Размер трафарета: 44,5х59,5<br>Материал: бумага<br>Количество в наборе: 6<br>Цвет: белый',
  'weight': None,
  'characteristics': []},
 {'id': 107736,
  'cod_good': '1000794132',
  'name': 'трафарет Снежинки 21х30см белый',
  'marketing_description': None,
  'url_key': '/catalog/trafarety-na-okna/1000794132/',
  'price': 0,
  'currency': 'RUB',
  'brand': {'id': 37, 'name': 'KAEMINGK'},
  'collection': None,
  'country': {'id': 1881, 'name': 'Китай'},
  'categories': {'225': 'Трафареты на окна'},
  'unit': None,
  'description': 'Название: Снежинки<br>Ра

In [8]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def medoids(products, k=3):
    texts = [f"{p['name']} {(p.get('description') or '')[:100]}" for p in products]
    X = TfidfVectorizer().fit_transform(texts)
    S = cosine_similarity(X)
    # medoid = строка с максимальной суммой сходств
    scores = S.sum(axis=1)
    idx = np.argsort(scores)[::-1][:k]
    return [products[i] for i in idx]

In [9]:
def get_medoids_from_cat(target_cat):
    products = []
    for item in data:
        catalog_cat = list(item["categories"].values())[0]
        if catalog_cat == target_cat:
            products.append(item)
    return medoids(products)

In [10]:
from tqdm import tqdm


def get_requests_batch(cats):
    medoids_batch = []

    for cat in tqdm(cats, desc="Computing medioids..."):
        medoids_batch.append(get_medoids_from_cat(cat))

    requests_batch = []
    for cat, medoids in tqdm(zip(cats, medoids_batch), desc="Formatting requests..."):
        requests_batch.append(get_choice_request(cat, medoids))

    return requests_batch

In [11]:
sample_cats = np.random.choice(un_cats, size=10)
sample_cats

array(['Лопаты совковые', 'Сменные блоки для тетрадей',
       'Тетради для записи иностранных слов',
       'Чехлы для досок гладильных', 'Крепления для гамаков',
       'Распылители масла', 'Выпрямители для волос',
       'Кувшины пластмассовые', 'Профили специальные',
       'Скатерти текстильные'], dtype='<U60')

In [12]:
requests_batch = get_requests_batch(sample_cats)

Computing medioids...: 100%|██████████| 10/10 [00:00<00:00, 41.56it/s]
Formatting requests...: 10it [00:00, 116185.71it/s]


In [13]:
out_batch = await judge.judge_batch(requests_batch)

In [14]:
requests_batch[-1]

{'state': 'Категория каталога: Скатерти текстильные\nПримеры товаров из каталога:\nУникальные скатерти изготовлены с применением последних технологических новинок: антискользящим напылением и пятнозащитной пропиткой. Мягкая жаккардовая ткань прекрасно обрамляет стол и делает его уютным на кухне, торжественным в банкетном зале, стильным в столовой и гостиной. Пятнозащитная пропитка обеспечит легкий уход и долгий срок службы.\nУникальные скатерти изготовлены с применением последних технологических новинок: антискользящим напылением и пятнозащитной пропиткой. Мягкая жаккардовая ткань прекрасно обрамляет стол и делает его уютным на кухне, торжественным в банкетном зале, стильным в столовой и гостиной. Пятнозащитная пропитка обеспечит легкий уход и долгий срок службы.\nУникальные скатерти изготовлены с применением последних технологических новинок: антискользящим напылением и пятнозащитной пропиткой. Мягкая жаккардовая ткань прекрасно обрамляет стол и делает его уютным на кухне, торжественн

In [ ]:
out_batch[-1]

{'model': 'FRIDA-Decisions',
 'answers': {'construction': {'type': 'choice',
   'probabilities': {'core': 0.22711833990110344,
    'adjacent': 0.6172621544775828,
    'excluded': 0.15561950562131374},
   'confidence': 0.15897127282554235,
   'choice': 'adjacent'}},
 'margins': {'construction': {'core': 2.7536349296569824,
   'adjacent': 3.753457546234131,
   'excluded': 2.375577688217163}},
 'usage': {'state_tokens': 165,
  'state_truncated': False,
  'questions': 1,
  'candidates': 3,
  'rows': 10,
  'encoder_tokens': 4142,
  'state_cache': 'off',
  'backend': 'torch',
  'milliseconds': 1158.57}}

In [15]:
requests_batch = get_requests_batch(un_cats)

Computing medioids...: 100%|██████████| 2375/2375 [00:54<00:00, 43.60it/s]
Formatting requests...: 2375it [00:00, 434429.66it/s]


In [16]:
import torch

torch.cuda.empty_cache()

In [17]:
from tqdm.asyncio import tqdm


def postprocessing_batch(out_batch):
    outs = []
    for out in out_batch:
        probs = out["answers"]["construction"]["probabilities"]
        label = max(probs, key=probs.get)
        outs.append(label)

    return outs

async def arun(request_batch):
    out_judge = await judge.judge_batch(request_batch)
    return postprocessing_batch(out_judge)

async def run_batch(requests_batch, batch_size=64):
    batches = [arun(requests_batch[i : i + batch_size]) for i in range(0, len(requests_batch), batch_size)]
    return await tqdm.gather(*batches)

In [18]:
out_batch = await run_batch(requests_batch)

100%|██████████| 38/38 [29:56<00:00, 47.27s/it]   


In [27]:
len(out_batch[0])

64

In [28]:
flatten_outs = [item for sublist in out_batch for item in sublist]
len(flatten_outs), len(un_cats)

(2375, 2375)

In [30]:
catalog_mapping = {str(cat): label for cat, label in zip(un_cats, flatten_outs)}

In [36]:
rel_keys = [key for key, value in catalog_mapping.items() if value != "excluded"]
len(rel_keys)

1369

In [40]:
import numpy as np

np.random.choice(rel_keys, size=10)

array(['Ванночки для краски', 'Краски аэрозольные', 'Лопаты штыковые',
       'Сетевые фильтры', 'Комплектующие для моек',
       'Скобы и гвозди для пневмостеплеров',
       'Фурнитура для деревянных окон',
       'Коврики и наклейки антискользящие для ванной',
       'Ножи и головки для триммеров и кос', 'Ленты малярные, клейкие'],
      dtype='<U60')

In [21]:
import numpy as np

ul, uc = np.unique(out_batch, return_counts=True)

for l, c in zip(ul, uc):
    print(f"{l}: {c}")

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (38,) + inhomogeneous part.

In [ ]:
import json

with open("catalog_mapping_filtering.json", "w", encoding="utf-8") as f:
    json.dump(catalog_mapping, f, indent=4, ensure_ascii=False)

{'% Аксессуары для бассейнов': ['excluded',
  'excluded',
  'excluded',
  'excluded',
  'core',
  'adjacent',
  'excluded',
  'excluded',
  'core',
  'core',
  'core',
  'excluded',
  'excluded',
  'adjacent',
  'excluded',
  'adjacent',
  'core',
  'core',
  'core',
  'core',
  'excluded',
  'core',
  'adjacent',
  'core',
  'excluded',
  'core',
  'core',
  'adjacent',
  'core',
  'core',
  'excluded',
  'excluded',
  'excluded',
  'adjacent',
  'excluded',
  'excluded',
  'excluded',
  'excluded',
  'adjacent',
  'excluded',
  'excluded',
  'excluded',
  'adjacent',
  'excluded',
  'adjacent',
  'excluded',
  'excluded',
  'adjacent',
  'core',
  'adjacent',
  'core',
  'adjacent',
  'core',
  'adjacent',
  'excluded',
  'adjacent',
  'excluded',
  'adjacent',
  'excluded',
  'excluded',
  'adjacent',
  'adjacent',
  'excluded',
  'adjacent'],
 '% Гамаки и подставки': ['excluded',
  'adjacent',
  'core',
  'adjacent',
  'core',
  'core',
  'excluded',
  'core',
  'adjacent',
  'excl